<a href="https://colab.research.google.com/github/arturgabriels779-alt/SISTEMA-PROFISSIONAL---MERCADO-CENTRAL-BH-v4.3/blob/main/mercado_sistema_entrega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# SISTEMA PROFISSIONAL - MERCADO CENTRAL BH v4.3 (Correção do Link)
!pip install streamlit plotly pyngrok pandas openpyxl -q

import os
import time
import threading
from datetime import datetime
import random

# 1. LIMPEZA
for arquivo in ["encomendas.db", "app.py", "encomendas.db-journal", "encomendas.db-wal", "encomendas.db-shm"]:
    if os.path.exists(arquivo):
        try: os.remove(arquivo)
        except: pass

# 2. CÓDIGO DO APP
codigo_app = '''
import streamlit as st
import sqlite3
import pandas as pd
from datetime import datetime
import plotly.express as px

# ========== CONFIGURAÇÃO ==========
st.set_page_config(page_title="Mercado Central BH", page_icon="🍎", layout="wide", initial_sidebar_state="collapsed")

# ========== CSS iOS ==========
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
    * { font-family: 'Inter', -apple-system, sans-serif; }
    .ios-header { background: rgba(255, 255, 255, 0.95); backdrop-filter: blur(20px); padding: 25px; border-radius: 24px; margin-bottom: 25px; box-shadow: 0 4px 30px rgba(0,0,0,0.05); text-align: center; }
    .ios-title { font-size: 2.4rem; font-weight: 800; background: linear-gradient(135deg, #007AFF 0%, #5856D6 100%); -webkit-background-clip: text; -webkit-text-fill-color: transparent; margin: 0; }
    .ios-card { background: white; border-radius: 20px; padding: 24px; margin-bottom: 20px; box-shadow: 0 2px 20px rgba(0,0,0,0.04); border: 1px solid #F5F5F7; }
    .ios-metric-card { text-align: center; padding: 15px; }
    .ios-metric-value { font-size: 2.2rem; font-weight: 800; line-height: 1; }
    .ios-metric-label { font-size: 0.9rem; color: #8E8E93; font-weight: 500; margin-top: 5px; }
    .ios-badge { display: inline-block; padding: 6px 14px; border-radius: 20px; font-size: 0.8rem; font-weight: 700; }
    .badge-blue { background: #E5F1FF; color: #007AFF; }
    .badge-green { background: #E8F5E9; color: #2E7D32; }
    .badge-orange { background: #FFF8E1; color: #F57C00; }
    .ios-section-title { font-size: 1.4rem; font-weight: 700; color: #1C1C1E; margin: 30px 0 20px 0; padding-left: 15px; border-left: 5px solid #007AFF; }
    .stButton>button { border-radius: 14px; font-weight: 600; height: 48px; }
</style>
""", unsafe_allow_html=True)

# ========== BANCO DE DADOS ==========
def get_db():
    conn = sqlite3.connect("encomendas.db", isolation_level=None)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_db()
    c = conn.cursor()
    c.execute("CREATE TABLE IF NOT EXISTS clientes (id INTEGER PRIMARY KEY AUTOINCREMENT, nome TEXT NOT NULL, telefone TEXT, loja TEXT NOT NULL, corredor TEXT NOT NULL, email TEXT, completo INTEGER DEFAULT 0, ativo INTEGER DEFAULT 1)")
    c.execute("CREATE TABLE IF NOT EXISTS encomendas (id TEXT PRIMARY KEY, numero_rastreio TEXT UNIQUE NOT NULL, data_chegada TIMESTAMP DEFAULT CURRENT_TIMESTAMP, logista_id INTEGER, nome_cliente_manual TEXT, loja TEXT NOT NULL, corredor TEXT NOT NULL, volumes INTEGER DEFAULT 1, status TEXT DEFAULT 'CENTRAL DE SEGURANÇA', data_retirada TIMESTAMP, quem_retirou TEXT, documento TEXT, justificativa_devolucao TEXT, responsavel TEXT NOT NULL, observacoes TEXT, transportadora TEXT, peso_aproximado TEXT)")

    if c.execute("SELECT COUNT(*) FROM clientes").fetchone()[0] == 0:
        clientes = [('Mercado Mineiro', '31987654321', 'Loja 101', 'A', 'contato@teste.com', 1)]
        c.executemany("INSERT INTO clientes (nome, telefone, loja, corredor, email, completo, ativo) VALUES (?,?,?,?,?,?,1)", clientes)
        conn.commit()
    conn.close()

if 'db_ok' not in st.session_state:
    init_db()
    st.session_state['db_ok'] = True

# ========== NAVEGAÇÃO ==========
st.markdown('<div class="ios-header"><h1 class="ios-title">🍎 Mercado Central BH</h1></div>', unsafe_allow_html=True)

menu = {"🏠 Início": "home", "📦 Nova": "nova", "✅ Status": "status", "👥 Clientes": "clientes", "📊 Relatórios": "relatorios"}
cols = st.columns(len(menu))
for i, (nome, chave) in enumerate(menu.items()):
    if cols[i].button(nome, use_container_width=True, key=f"nav_{chave}"):
        st.session_state['page'] = chave
        st.rerun()

if 'page' not in st.session_state: st.session_state['page'] = 'home'
page = st.session_state['page']

# ========== FUNÇÕES ==========
def badge(status):
    cores = {'CENTRAL DE SEGURANÇA': 'badge-blue', 'ENTREGUE': 'badge-green', 'AGUARDANDO DEVOLUÇÃO': 'badge-orange'}
    return f"<span class='ios-badge {cores.get(status, 'badge-gray')}'>{status}</span>"

# ========== HOME (COM BOTÃO EDITAR) ==========
if page == "home":
    st.markdown('<div class="ios-section-title">📊 Visão Geral</div>', unsafe_allow_html=True)
    conn = get_db()
    central = conn.execute("SELECT COUNT(*) FROM encomendas WHERE status='CENTRAL DE SEGURANÇA'").fetchone()[0]
    entregues = conn.execute("SELECT COUNT(*) FROM encomendas WHERE status='ENTREGUE'").fetchone()[0]
    conn.close()

    c1, c2 = st.columns(2)
    c1.markdown(f"<div class='ios-card ios-metric-card'><div class='ios-metric-value' style='color:#007AFF'>{central}</div><div class='ios-metric-label'>🛡️ Na Central</div></div>", unsafe_allow_html=True)
    c2.markdown(f"<div class='ios-card ios-metric-card'><div class='ios-metric-value' style='color:#34C759'>{entregues}</div><div class='ios-metric-label'>✅ Entregues</div></div>", unsafe_allow_html=True)

    st.markdown('<div class="ios-section-title">🕓 Últimas Encomendas</div>', unsafe_allow_html=True)
    conn = get_db()
    recentes = conn.execute("SELECT e.*, c.nome as cliente_nome FROM encomendas e LEFT JOIN clientes c ON e.logista_id = c.id ORDER BY e.rowid DESC LIMIT 10").fetchall()
    conn.close()

    for r in recentes:
        nome_exibicao = r['cliente_nome'] or r['nome_cliente_manual'] or "N/A"
        col_info, col_btn = st.columns([5, 1])

        with col_info:
            st.markdown(f"""
                <div class='ios-card' style='margin-bottom:0px; padding:15px;'>
                    <div style="font-weight:700;">📦 {r['numero_rastreio']} <span style="font-weight:400; font-size:0.9rem; color:#666;">({nome_exibicao})</span></div>
                    <div style="color:#888; font-size:0.85rem;">📍 {r['loja']} • {r['corredor']} | {badge(r['status'])}</div>
                </div>
            """, unsafe_allow_html=True)

        with col_btn:
            if st.button("➡️", key=f"go_{r['id']}", help="Editar Encomenda"):
                st.session_state['buscar_id'] = r['numero_rastreio']
                st.session_state['page'] = 'status'
                st.rerun()
        st.write("")

# ========== NOVA ENCOMENDA ==========
elif page == "nova":
    st.markdown('<div class="ios-section-title">📦 Nova Encomenda</div>', unsafe_allow_html=True)

    st.markdown('<div class="ios-card">', unsafe_allow_html=True)
    tipo = st.radio("Tipo de Destinatário", ["Cliente Cadastrado", "Cliente Avulso"], horizontal=True)

    conn = get_db()
    clientes_db = conn.execute("SELECT id, nome, loja, corredor, completo FROM clientes WHERE ativo=1 ORDER BY nome").fetchall()
    conn.close()

    cliente_sel_id = None
    loja_auto = ""
    corredor_auto = ""

    if tipo == "Cliente Cadastrado":
        opcoes = {f"{c['nome']} ({c['loja']})": c for c in clientes_db}
        sel = st.selectbox("Selecione o Cliente", [""] + list(opcoes.keys()))
        if sel:
            cliente = opcoes[sel]
            cliente_sel_id = cliente['id']
            loja_auto = cliente['loja']
            corredor_auto = cliente['corredor']
            if cliente['completo'] == 0:
                st.warning("⚠️ Cadastro incompleto. Atualize na aba Clientes.")
    else:
        st.info("📝 Preencha os dados abaixo. O cliente será cadastrado automaticamente.")

    st.markdown('</div>', unsafe_allow_html=True)

    with st.form("form_nova"):
        col1, col2 = st.columns(2)
        rastreio = col1.text_input("Nº Rastreio *")
        transp = col2.selectbox("Transportadora *", ["Correios", "Jadlog", "Loggi", "Total Express", "Outra"])

        nome_avulso = ""
        tel_avulso = ""
        if tipo == "Cliente Avulso":
            c_nome, c_tel = st.columns([3, 1])
            nome_avulso = c_nome.text_input("Nome do Cliente *")
            tel_avulso = c_tel.text_input("Telefone")

        col_l, col_c = st.columns(2)
        loja = col_l.text_input("Loja/Box *", value=loja_auto)
        corredor = col_c.text_input("Corredor *", value=corredor_auto)

        col_v, col_p, col_r = st.columns(3)
        volumes = col_v.number_input("Volumes", min_value=1, value=1)
        peso = col_p.selectbox("Peso", ["Não informado", "Até 5kg", "5-15kg", "+15kg"])
        resp = col_r.selectbox("Recebido por *", ["Carlos Silva", "Ana Paula", "Roberto Costa", "Mariana Lima"])

        obs = st.text_area("Observações")

        if st.form_submit_button("💾 Salvar Encomenda", type="primary", use_container_width=True):
            if not rastreio or not loja or not corredor or (tipo == "Cliente Avulso" and not nome_avulso):
                st.error("❌ Preencha todos os campos obrigatórios (*).")
            else:
                try:
                    conn = get_db()
                    c = conn.cursor()
                    final_cliente_id = cliente_sel_id

                    if tipo == "Cliente Avulso":
                        existente = c.execute("SELECT id FROM clientes WHERE nome = ? AND loja = ?", (nome_avulso, loja)).fetchone()
                        if existente:
                            final_cliente_id = existente[0]
                            st.info(f"ℹ️ Cliente '{nome_avulso}' já existente. Vinculado.")
                        else:
                            completo = 1 if tel_avulso else 0
                            c.execute("INSERT INTO clientes (nome, telefone, loja, corredor, email, completo, ativo) VALUES (?,?,?,?,?,?,1)",
                                     (nome_avulso, tel_avulso, loja, corredor, "", completo))
                            final_cliente_id = c.lastrowid
                            st.success(f"✅ Novo cliente '{nome_avulso}' cadastrado!")

                    novo_id = f"MCBH-{datetime.now().strftime('%Y%m%d%H%M%S')}"
                    c.execute("INSERT INTO encomendas (id, numero_rastreio, logista_id, loja, corredor, volumes, responsavel, transportadora, peso_aproximado, observacoes) VALUES (?,?,?,?,?,?,?,?,?,?)",
                             (novo_id, rastreio, final_cliente_id, loja, corredor, volumes, resp, transp, peso, obs))

                    conn.commit()
                    conn.close()
                    st.success(f"✅ Encomenda {rastreio} registrada!")
                    st.balloons()

                except sqlite3.IntegrityError:
                    st.error("❌ Erro: Este número de rastreio já existe.")
                except Exception as e:
                    st.error(f"Erro: {e}")

# ========== STATUS ==========
elif page == "status":
    st.markdown('<div class="ios-section-title">✅ Atualizar Status</div>', unsafe_allow_html=True)

    default_val = st.session_state.pop('buscar_id', "")
    busca = st.text_input("🔍 Buscar Rastreio ou ID", value=default_val)

    if busca:
        conn = get_db()
        res = conn.execute("SELECT e.*, c.nome as cliente_nome FROM encomendas e LEFT JOIN clientes c ON e.logista_id = c.id WHERE e.numero_rastreio=? OR e.id=?", (busca,busca)).fetchone()
        conn.close()
        if res:
            st.session_state['enc_atual'] = dict(res)

    if 'enc_atual' in st.session_state and st.session_state['enc_atual']:
        enc = st.session_state['enc_atual']
        nome_cli = enc['cliente_nome'] or enc['nome_cliente_manual'] or "N/A"

        st.markdown(f"""
            <div class='ios-card'>
                <h3>{enc['numero_rastreio']}</h3>
                <p>👤 {nome_cli} | 📍 {enc['loja']} - {enc['corredor']}</p>
            </div>
        """, unsafe_allow_html=True)

        novo_status = st.selectbox("Novo Status", ["CENTRAL DE SEGURANÇA", "ENTREGUE", "AGUARDANDO DEVOLUÇÃO", "DEVOLVIDO"])

        with st.form("form_status"):
            quem = None; doc = None; motivo = None
            if novo_status == "ENTREGUE":
                quem = st.text_input("Quem Retirou *")
                doc = st.text_input("Documento *")
            elif novo_status == "DEVOLVIDO":
                motivo = st.selectbox("Motivo", ["Cliente não encontrado", "Endereço errado", "Recusado"])

            obs = st.text_area("Observações", value=enc['observacoes'] or "")

            if st.form_submit_button("💾 Atualizar", type="primary"):
                conn = get_db()
                if novo_status == "ENTREGUE" and quem and doc:
                    conn.execute("UPDATE encomendas SET status=?, data_retirada=datetime('now'), quem_retirou=?, documento=?, observacoes=? WHERE id=?",
                               (novo_status, quem, doc, obs, enc['id']))
                elif novo_status == "DEVOLVIDO":
                    conn.execute("UPDATE encomendas SET status=?, justificativa_devolucao=?, observacoes=? WHERE id=?",
                               (novo_status, motivo, obs, enc['id']))
                else:
                    conn.execute("UPDATE encomendas SET status=?, observacoes=? WHERE id=?", (novo_status, obs, enc['id']))
                conn.commit()
                conn.close()
                st.success("✅ Atualizado!")
                del st.session_state['enc_atual']
                st.rerun()

# ========== CLIENTES (COM ATUALIZAÇÃO EM MASSA) ==========
elif page == "clientes":
    st.markdown('<div class="ios-section-title">👥 Clientes</div>', unsafe_allow_html=True)

    editing_id = st.session_state.get('edit_cliente_id', None)

    if editing_id:
        conn = get_db()
        cliente_atual = conn.execute("SELECT * FROM clientes WHERE id = ?", (editing_id,)).fetchone()
        conn.close()

        if cliente_atual:
            st.markdown('<div class="ios-card">', unsafe_allow_html=True)
            st.subheader(f"✏️ Editando: {cliente_atual['nome']}")

            with st.form("form_edit_cliente"):
                n_nome = st.text_input("Nome", value=cliente_atual['nome'])
                n_tel = st.text_input("Telefone", value=cliente_atual['telefone'])
                n_loja = st.text_input("Loja", value=cliente_atual['loja'])
                n_corr = st.text_input("Corredor", value=cliente_atual['corredor'])
                n_email = st.text_input("Email", value=cliente_atual['email'])

                atualizar_tudo = st.checkbox("🔄 Aplicar alterações de endereço (Loja/Corredor) a TODAS as encomendas deste cliente", value=False)

                col_b1, col_b2 = st.columns(2)
                with col_b1:
                    if st.form_submit_button("💾 Salvar Alterações", type="primary"):
                        conn = get_db()
                        c = conn.cursor()
                        c.execute("UPDATE clientes SET nome=?, telefone=?, loja=?, corredor=?, email=?, completo=? WHERE id=?",
                                 (n_nome, n_tel, n_loja, n_corr, n_email, 1 if n_email and n_tel else 0, editing_id))

                        if atualizar_tudo:
                            c.execute("UPDATE encomendas SET loja=?, corredor=? WHERE logista_id=?", (n_loja, n_corr, editing_id))
                            st.info(f"✅ Endereço atualizado em todas as encomendas de {n_nome}.")

                        conn.commit()
                        conn.close()
                        st.success("Cliente atualizado com sucesso!")
                        st.session_state['edit_cliente_id'] = None
                        st.rerun()

                with col_b2:
                    if st.form_submit_button("❌ Cancelar"):
                        st.session_state['edit_cliente_id'] = None
                        st.rerun()

            st.markdown('</div>', unsafe_allow_html=True)

    else:
        conn = get_db()
        clientes = conn.execute("SELECT * FROM clientes WHERE ativo=1 ORDER BY nome").fetchall()
        conn.close()

        for c in clientes:
            col_info, col_edit = st.columns([5, 1])
            icon = "✅" if c['completo'] else "⚠️"

            with col_info:
                st.markdown(f"""
                    <div class='ios-card' style='margin-bottom:0px; padding:15px;'>
                        <b>{c['nome']}</b> {icon}<br>
                        <span style='color:#666'>📍 {c['loja']} | {c['corredor']} | 📱 {c['telefone']}</span>
                    </div>
                """, unsafe_allow_html=True)

            with col_edit:
                if st.button("✏️", key=f"edit_btn_{c['id']}", help="Editar Cliente"):
                    st.session_state['edit_cliente_id'] = c['id']
                    st.rerun()
            st.write("")

# ========== RELATÓRIOS ==========
elif page == "relatorios":
    st.markdown('<div class="ios-section-title">📊 Relatórios</div>', unsafe_allow_html=True)
    conn = get_db()
    df = pd.read_sql_query("SELECT e.id, e.numero_rastreio, c.nome as cliente, e.loja, e.status, e.responsavel FROM encomendas e LEFT JOIN clientes c ON e.logista_id = c.id", conn)
    conn.close()
    st.dataframe(df, use_container_width=True)
    csv = df.to_csv(index=False).encode('utf-8')
    st.download_button("📥 Baixar CSV", data=csv, file_name="relatorio.csv")
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(codigo_app)

print("✅ Sistema v4.3 gerado.")

# 3. EXECUÇÃO
def run():
    os.system("streamlit run app.py --server.port 8501 --server.headless true")

t = threading.Thread(target=run)
t.start()
time.sleep(5)

# 4. NGROK (CORREÇÃO DO PRINT)
from pyngrok import ngrok, conf
conf.get_default().auth_token = "38ybdCOek4nliyp86IxOLXqGstw_5RBMbbZCPKeyvkTCwaVgb"

try:
    ngrok.kill()
    url = ngrok.connect(8501, "http")
    # Correção aqui: usando f-string corretamente
    print("\n" + "="*60)
    print(f"🚀 SISTEMA ONLINE (v4.3): {url}")
    print("="*60)
    print("\nNOVIDADES:")
    print("1. Na tela Início, clique em '➡️' para editar o pedido.")
    print("2. Na aba Clientes, clique em '✏️' para editar.")
    print("3. Marque o checkbox para atualizar endereço de todas as encomendas.")
except Exception as e:
    print(f"Erro ao gerar link: {e}")

✅ Sistema v4.3 gerado.

🚀 SISTEMA ONLINE (v4.3): NgrokTunnel: "https://abysmally-judicable-veronica.ngrok-free.dev" -> "http://localhost:8501"

NOVIDADES:
1. Na tela Início, clique em '➡️' para editar o pedido.
2. Na aba Clientes, clique em '✏️' para editar.
3. Marque o checkbox para atualizar endereço de todas as encomendas.
